# CMU 350 — Scraping de texto con Regex

## El problema
Antes, HTML nos regalaba estructura:

`.title` · `.venue` · `.price`

Ahora tenemos **texto**. Sabemos qué buscamos, pero no dónde está.

En esta mini-demostración intentaremos recuperar:

- hora
- precio

y después veremos por qué las reglas empiezan a complicarse.

## 1. Cargar el texto desde la página

Abre primero `agenda-cultural-pr-texto.html` en el navegador.

Luego extraemos el texto visible. Aquí BeautifulSoup **no identifica eventos ni campos**; solamente elimina las etiquetas HTML.

In [ ]:
from pathlib import Path
from bs4 import BeautifulSoup
import re
import pandas as pd

html = Path("agenda-cultural-pr-texto.html").read_text(encoding="utf-8")
soup = BeautifulSoup(html, "html.parser")

text = soup.get_text("\n", strip=True)
print(text[:1500])

## 2. Buscar algo que tiene una forma reconocible

Una hora suele tener una forma como:

`7:30 PM`  
`10:00 AM`

Eso puede describirse mediante una **expresión regular**.

In [ ]:
time_pattern = r"\b\d{1,2}:\d{2}\s*(?:AM|PM)\b"

times = re.findall(time_pattern, text, flags=re.IGNORECASE)
times

### Lee el patrón

- `\d{1,2}` → uno o dos dígitos
- `:` → dos puntos
- `\d{2}` → dos dígitos
- `AM|PM` → una de esas dos alternativas

La computadora no sabe qué es *la hora de un evento*. Solo reconoce una forma textual.

## 3. Ahora: precios

Empecemos ingenuamente buscando un signo `$` seguido de un número.

In [ ]:
price_pattern_v1 = r"\$\d+(?:\.\d{2})?"

prices_v1 = re.findall(price_pattern_v1, text)
prices_v1

## 4. ¿Qué perdimos?

Mira la página otra vez.

Hay eventos que dicen:

- `Entrada gratis`
- `FREE`
- `Entrada libre`
- `No se cobra entrada`

Nuestro patrón de `$` no los reconoce.

Hagamos una regla más inteligente.

In [ ]:
price_pattern_v2 = r"\$\d+(?:\.\d{2})?|gratis|free|entrada libre|no se cobra entrada"

prices_v2 = re.findall(price_pattern_v2, text, flags=re.IGNORECASE)
prices_v2

### ¿Ya resolvimos el problema?

No exactamente.

Busca este evento:

**Laboratorio de podcast — Precio: $15-25**

Nuestra expresión encuentra `$15`, pero no entiende que el texto expresa **un rango**.

Esta es la idea importante:

> Cada nueva forma de escribir la misma información puede obligarnos a modificar nuestras reglas.

In [ ]:
price_pattern_v3 = r"\$\d+(?:\.\d{2})?(?:-\d+(?:\.\d{2})?)?|gratis|free|entrada libre|no se cobra entrada"

prices_v3 = re.findall(price_pattern_v3, text, flags=re.IGNORECASE)
prices_v3

## 5. Mini experimento: convertir texto en registros

Para mantener este ejemplo sencillo, el documento separa cada evento con una línea en blanco.

Podemos aprovechar **esa convención** para crear bloques. Esto ya es una decisión/hipótesis del scraper.

In [ ]:
# Recuperamos solamente el contenido del <pre>, que contiene los eventos.
raw = soup.find("pre").get_text()

blocks = [b.strip() for b in re.split(r"\n\s*\n", raw) if b.strip()]
blocks = blocks[1:]  # elimina el encabezado
len(blocks)

In [ ]:
records = []

for block in blocks:
    lines = [line.strip() for line in block.splitlines() if line.strip()]
    time_match = re.search(time_pattern, block, flags=re.IGNORECASE)
    price_match = re.search(price_pattern_v3, block, flags=re.IGNORECASE)

    records.append({
        "title_guess": lines[0],
        "time": time_match.group(0) if time_match else None,
        "price_text": price_match.group(0) if price_match else None
    })

df = pd.DataFrame(records)
df

## 6. Detente aquí: ¿qué hizo realmente nuestro scraper?

Funcionó porque nosotros asumimos varias cosas:

1. cada evento está separado por una línea vacía;
2. la primera línea es el título;
3. una hora tiene una forma que conocemos;
4. un precio pertenece a una lista de formas que anticipamos.

**PREGUNTA:** ¿Qué ocurriría si alguien escribiera:

> Entrada general: $15. Estudiantes: $8. Menores de 12 años: gratis.

¿Cuál es *el precio*?

Quizá el problema ya no sea escribir una regex mejor.

Quizá nuestro campo `price` sea demasiado simple.

## Puente hacia el próximo experimento

Un scraper basado en reglas es:

**predecible + explicable + reproducible**

pero también:

**rígido + dependiente de patrones + frágil ante variaciones**

La próxima pregunta es:

> ¿Qué ocurre si entregamos exactamente este mismo texto a una IA y le pedimos que construya la tabla?